# RiskOptima Credit Risk Model Demo

This notebook demonstrates a synthetic PD/LGD/EAD obligor portfolio, expected loss, unexpected loss, simulated credit loss distribution, Credit VaR/CVaR, rating migration, and Merton structural default probability.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from riskoptima.credit import (
    expected_loss,
    unexpected_loss,
    portfolio_expected_loss,
    simulate_credit_losses,
    credit_var,
    credit_cvar,
    validate_transition_matrix,
    simulate_rating_migration,
    merton_distance_to_default,
    merton_pd,
)

## Synthetic Obligor Portfolio

In [ ]:
rng = np.random.default_rng(42)
n_obligors = 100
portfolio = pd.DataFrame({
    "obligor": [f"OBL{i:03d}" for i in range(n_obligors)],
    "rating": rng.choice(["AAA", "AA", "A", "BBB", "BB", "B"], size=n_obligors, p=[0.05, 0.10, 0.25, 0.30, 0.20, 0.10]),
    "PD": rng.uniform(0.002, 0.08, size=n_obligors),
    "LGD": rng.uniform(0.25, 0.65, size=n_obligors),
    "EAD": rng.lognormal(mean=13.0, sigma=0.5, size=n_obligors),
})
portfolio.head()

## Expected and Unexpected Loss

In [ ]:
portfolio["expected_loss"] = expected_loss(portfolio["PD"], portfolio["LGD"], portfolio["EAD"])
portfolio["unexpected_loss"] = unexpected_loss(portfolio["PD"], portfolio["LGD"], portfolio["EAD"], asset_correlation=0.2)

summary = pd.Series({
    "portfolio_expected_loss": portfolio_expected_loss(portfolio),
    "portfolio_unexpected_loss_approx": portfolio["unexpected_loss"].sum(),
    "total_ead": portfolio["EAD"].sum(),
})
summary

## Simulated Loss Distribution and Credit VaR / CVaR

In [ ]:
losses = simulate_credit_losses(portfolio, n_sims=20_000, random_state=123)
var_99 = credit_var(losses, confidence=0.99)
cvar_99 = credit_cvar(losses, confidence=0.99)
pd.Series({"Credit VaR 99%": var_99, "Credit CVaR 99%": cvar_99})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(losses, bins=50, alpha=0.75)
ax.axvline(var_99, color="red", linestyle="--", label="VaR 99%")
ax.axvline(cvar_99, color="black", linestyle="-", label="CVaR 99%")
ax.set_title("Simulated Credit Loss Distribution")
ax.set_xlabel("Loss")
ax.legend();

## Rating Migration Simulation

In [ ]:
ratings = ["AAA", "AA", "A", "BBB", "BB", "B", "D"]
transition_matrix = pd.DataFrame([
    [0.90, 0.08, 0.02, 0.00, 0.00, 0.00, 0.00],
    [0.02, 0.88, 0.08, 0.02, 0.00, 0.00, 0.00],
    [0.00, 0.03, 0.87, 0.08, 0.01, 0.00, 0.01],
    [0.00, 0.00, 0.04, 0.84, 0.08, 0.02, 0.02],
    [0.00, 0.00, 0.00, 0.05, 0.80, 0.10, 0.05],
    [0.00, 0.00, 0.00, 0.00, 0.08, 0.78, 0.14],
    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00],
], index=ratings, columns=ratings)

validate_transition_matrix(transition_matrix)
migration_paths = simulate_rating_migration(portfolio["rating"].head(10), transition_matrix, periods=5, random_state=42)
migration_paths

## Merton Distance to Default

In [ ]:
asset_value = 150
debt_face_value = 100
asset_vol = 0.25
risk_free_rate = 0.03
maturity = 1.0

pd.Series({
    "distance_to_default": merton_distance_to_default(asset_value, debt_face_value, asset_vol, risk_free_rate, maturity),
    "merton_pd": merton_pd(asset_value, debt_face_value, asset_vol, risk_free_rate, maturity),
})